# Feature Extraction on CIFAR‑10
Esta prática demonstra como usar um modelo pré‑treinado (ResNet‑18) para extrair recursos de imagens e treinar apenas a camada final.
*Objetivo:* treinar um classificador em ≤ 10 min.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Transformações básicas
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_dataset   = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

In [ ]:
# Carregar modelo pré‑treinado
model = models.resnet18(pretrained=True)
# Congelar todas as camadas convolucionais
for param in model.parameters():
    param.requires_grad = False
# Substituir a camada final (fc) para 10 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)
model = model.to('cpu')  # ou .to('cuda') se houver GPU

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=0.01, momentum=0.9)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
num_epochs = 5
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer)
    acc  = evaluate(model, val_loader)
    print(f'Epoch {epoch+1}/{num_epochs} – Loss: {loss:.4f} – Val Acc: {acc:.4f}')